In [ ]:
import csv

In [ ]:
def data_normalized(list_dt, mean=None, std=None):
    if mean is None or std is None:
        mean = [0 for i in range(len(list_dt[0])-1)]
        for i in range(len(list_dt)):
            for j in range(len(list_dt[i])-1):
                mean[j] += list_dt[i][j]
        for i in range(len(mean)):
            mean[i] /= len(list_dt)
        std = [0 for i in range(len(mean))]
        for i in range(len(list_dt)):
            for j in range(len(list_dt[i])-1):
                std[j] += (list_dt[i][j] - mean[j])**2
        for i in range(len(std)):
            std[i] /= len(list_dt)
            std[i] = std[i] ** 0.5
    else:
        mean = mean
        std = std
    new_list_dt = []
    for i in range(len(list_dt)):
        row_i = []
        for j in range(len(list_dt[i])-1):
            x_i_j = (list_dt[i][j]-mean[j])/std[j]
            row_i.append(x_i_j)
        row_i.append(list_dt[i][-1])
        new_list_dt.append(row_i)
    return new_list_dt, mean, std

Функция которая нормализует данные и любой диапазон меняет на диапазон от -1 до 1

In [ ]:
def prev_csv(data):
    list_data = []
    flag = 0
    for row in data:
        if flag == 0:
            flag+=1
        else:
            l = list(row)
            t = [float(i) for i in l]
            list_data.append(t)
    return list_data

Функция "превращает" csv в список с которым мне было удобней работать

In [ ]:
def gradient(w, b,   data, number_of_sings=19):
    grad = [0 for i in range(number_of_sings)]
    gradb = 0
    for i in range(len(data)):
        pred = b
        for j in range(number_of_sings):
            pred += w[j]*data[i][j]
        err = pred - data[i][-1]
        for j in range(number_of_sings):
            grad[j] += err*data[i][j]*2
        gradb += err*2
    for i in range(number_of_sings):
        grad[i] /= len(data)
    gradb /= len(data)
    return grad, gradb

Функция gradient считает градит от функции ошибки:
$$MSE=\frac{1}{n}\sum(y_i-y_{real})^2$$
Где $y_i$ это предсказанная цена, полученная перемножением текущих весов на параметры i-ой квартиры.

$y_{real}$ - реальная цена i-ой квартиры

$n$ - количество квартир

Стоит также отметить, что возвращается градиент от параметров и градиент от свободного члена отдельно

In [ ]:
def find_weight_momentum(data, number_of_epochs = 12, rate_learning = 0.05, rate_inertion = 0.3):
    x = [0.0 for i in range(19)]
    v = [0.0 for i in range(19)]
    v_b = 0
    b = sum(data[i][-1] for i in range(len(data)))/len(data)
    for i in range(number_of_epochs):
        g, gradb = gradient(x, b, data)
        for j in range(len(x)):
            v[j] = v[j]*rate_inertion- (rate_learning*g[j])
            x[j]+=v[j]
        v_b = v_b*rate_inertion- (rate_learning*gradb)
        b+=v_b
    return x, b

В этой функции реализована идея **тяжелого шарика** или же **momentum**. То есть помимо обычного градиента добавляется некоторый *момент инерции* быстрей сходится к минимуму и преодолевать локальные минимумы. Свою идею берет из физической аналогии, где шар скатывается с горки постепенно увеличивая скорость и преодолевая ямки за счет накопленной инерции. С нулевых весов сходится примерно за сто итераций

In [ ]:
def find_weight_Nesterov_momentum(data, number_of_epochs = 100, rate_learning = 0.05, rate_inertion = 0.3):
    x = [0.0 for i in range(19)]
    v = [0.0 for i in range(19)]
    b = sum(data[i][-1] for i in range(len(data)))/len(data)
    v_b = 0
    for i in range(number_of_epochs):
        new_x = [x[j]+v[j]*rate_inertion for j in range(len(x))]
        g, gradb = gradient(new_x, b, data)
        for j in range(len(x)):
            v[j] = v[j]*rate_inertion- (rate_learning*g[j])
            x[j]+=v[j]
        v_b = v_b*rate_inertion- (rate_learning*gradb)
        b+=v_b
    return x, b

Оптимизация предыдущего метода придуманная **Ю.Е.Нестеровым** в честь которого и названа. Оптимизация заключается в том, чтоб считать градиент не от текущих весов, а от тех весов в которые мы бы попали с текущей инерцией. Эта оптимизация значительно улучшает оценку по порядку и на практике лучше, чем предыдущий алгоритм.Сходится примерно за 90 итераций.

In [ ]:
def elipsoid_method(data, number_of_epochs = 500, radius = 150):
    elipsoid = [[0.0 for i in range(20)] for i in range(20)]
    center = [0 for i in range(20)]
    center[19] = 10
    for i in range(20):
        elipsoid[i][i] = radius
    for i in range(number_of_epochs):
        x = center[:19]
        b = center[19]
        g, gradb = gradient(x, b, data)
        g.append(gradb)
        start_h = g
        elipsoid1 = [0 for i in range(20)]
        for i in range(20):
            for j in range(20):
                elipsoid1[i] += elipsoid[i][j] * start_h[j]
        elipsoid2 = 0
        for i in range(20):
            elipsoid2 += elipsoid1[i] * start_h[i]
        h = [0 for i in range(20)]
        for i in range(20):
            h[i] = elipsoid1[i] / (elipsoid2 ** 0.5)

        for i in range(20):
            center[i] -= h[i] / 21

        for i in range(20):
            for j in range(20):
                elipsoid[i][j] -= (2 / 21) * h[i] * h[j]
                elipsoid[i][j] *= (400 / 399)
    return center[:19], center[19]

В этом алгоритме реализована другая физическая идея. Представьте себе многограник в котором есть некоторая точка котороя вам интересна, но вы не знаете где именно. Метод заключается в том, чтоб отсекать от этого многограника части в которых этой точки нет. После того как фигуру разрезали достаточное количество раз получится фигура с центром тяжести довольно близкой к точке из-за того что фигура сама будет достаточно мала.

Так вот если мы представим себе что в пространстве весов, где каждая координата соответствует весу определенного параметра, а многомерная фигура задает область в которой есть точка минимума применяя этот метод можно эту точку достаточно быстро найти. Идея следующая:
- находим центр тяжести этой фигуры
- считаем градиент от этой точки(точка в данном случае это веса)
- отсекаем ту часть, на которую указывает градиент
- повторяем предыдущие действия достаточное количество раз
- получаем фигуру с центром тяжести достаточно близким к точке минимума

Самый тяжелый пункт у этой задачи первый поэтому используем элипсоидный метод где фигуру заменяем на элипсоид таким образом чтоб элипсоид ее полностью ее покрывал и обновить центр масс становится значительно легче, но падает точность.
Сходится примерно за 500 итераций

In [ ]:
def errors(data, v, b):
    mse = 0
    mn = 0
    for i in range(len(data)):
        pred = b
        for j in range(len(v)):
            pred += v[j] * data[i][j]
        err = data[i][-1] - pred
        mse += err * err
        mn += data[i][-1]
    mse /= len(data)
    rmse = mse ** 0.5

    mn /= len(data)
    ss_total = 0
    ss_res = 0
    for i in range(len(data)):
        pred = b
        for j in range(len(v)):
            pred += v[j] * data[i][j]
        ss_total += (mn - data[i][-1]) ** 2
        ss_res += (data[i][-1] - pred) ** 2

    r = 1 - ss_res / ss_total

    print("mse:", mse)
    print("rmse:", rmse)
    print("r:", r)
    return mse, rmse, r

Считает насколько удачно подобраны веса и возвращает три вида ошибок MSE, RMSE, R^2